# 🏦 Loan Approval Prediction — EDA & Model Building

This notebook covers:
1. **Exploratory Data Analysis (EDA)** on loan applicant records
2. **Feature Engineering & Preprocessing** (Imputation, Standard Scaling, One-Hot Encoding)
3. **Random Forest Classifier Training** inside an end-to-end `scikit-learn Pipeline`
4. **Evaluation Metrics** (Accuracy, Confusion Matrix, ROC-AUC)
5. **Serialization** into `models/loan_approval_model.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Load Data
df = pd.read_csv('../data/loan_approval_dataset.csv')
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

print('Dataset Shape:', df.shape)
df.head()

In [ ]:
# 2. Target Distribution
print('Target Counts:')
print(df['loan_status'].value_counts())
print('\nPercentage:')
print(df['loan_status'].value_counts(normalize=True) * 100)

In [ ]:
# 3. Model Pipeline Setup
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import joblib

X = df.drop(columns=['loan_id', 'loan_status'])
y = df['loan_status'].map({'Approved': 1, 'Rejected': 0})

num_cols = [
    'no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score',
    'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value'
]
cat_cols = ['education', 'self_employed']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42))
])

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print('ROC-AUC :', round(roc_auc_score(y_test, y_prob) * 100, 2), '%')
print('\nClassification Report:\n', classification_report(y_test, y_pred))